# FloorPlan-AI: U-Net (ResNet-18) fine-tuning on CubiCasa5K — Colab runbook

**Runtime: T4 GPU** (Runtime > Change runtime type). Run the cells in order.

**What this notebook does.** Real gradient-based fine-tuning of a U-Net whose encoder is an ImageNet-pretrained ResNet-18, on real
CubiCasa5K floor plans, then evaluation on held-out test plans and polygon-JSON export.
It does **not** use synthetic data, Qwen2.5-VL or any pretrained-only inference.

**Two stages with a human gate between them**
* **Stage A (data):** download the official zip, inspect the REAL SVG structure, build masks and overlays.
  Look at the overlays. Download `stageA_for_review.zip` and send it to Claude for an independent look before training.
* **Stage B (model):** only after you set `OVERLAYS_CONFIRMED = True`: tests, overfit-8 sanity run, full training, evaluation, packaging.

Nothing in this notebook has been executed by its author (the authoring sandbox has no GPU, no torch and cannot reach Zenodo).
Any failure you hit is useful information: send the cell output.

## Stage A — data

In [ ]:
# A1. Clone the branch and install light dependencies (torch/torchvision are already on Colab)
import os, subprocess, sys
REPO = "/content/FloorPlan-AI"
if not os.path.isdir(REPO):
    !git clone https://github.com/sandhiya-1234/FloorPlan-AI.git {REPO}
os.chdir(REPO)
!git checkout repair/real-vlm-pipeline
!git pull --ff-only
!git log --oneline -1
!pip install -q -r requirements-seg.txt
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# A2. (optional) keep outputs across Colab disconnects: copy results to Google Drive at the end of each stage
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/floorplan_ai"
os.makedirs(DRIVE, exist_ok=True)
print("Drive folder:", DRIVE)

In [ ]:
# A3. Download the official CubiCasa5K zip (Zenodo record 2613548) and verify the checksum shown on the record page
import hashlib
ZIP = "/content/cubicasa5k.zip"
URL = "https://zenodo.org/records/2613548/files/cubicasa5k.zip?download=1"
EXPECTED_MD5 = "0ce0b203d1e3c125b51087b219bd23b9"   # from the Zenodo record page (file cubicasa5k.zip, 5.5 GB)
if not os.path.isfile(ZIP) or os.path.getsize(ZIP) < 5_000_000_000:
    !wget -c --tries=5 --timeout=60 -O {ZIP} "{URL}"
md5 = hashlib.md5()
with open(ZIP, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        md5.update(chunk)
print("size GB:", round(os.path.getsize(ZIP) / 1e9, 2), "| md5:", md5.hexdigest())
assert md5.hexdigest() == EXPECTED_MD5, "checksum mismatch: the download is incomplete or the file changed. Do not continue."
print("checksum OK")

In [ ]:
# A4. INSPECT the real archive: folder layout, official split lists, SVG group ids/classes, room names, selector presence.
# This exits non-zero (and the next cells must not be run) if an official selector never occurs in the real files.
os.makedirs("/content/stageA", exist_ok=True)
!python -m training.prepare_dataset inspect --zip {ZIP} --n 60 2>&1 | tee /content/stageA/inspect_report.txt

In [ ]:
# A5. Build the subset: seeded sample from the OFFICIAL train/val/test lists, masks from the explicit mapping,
# cross-split duplicate removal, overlays, manifest.  (Reads the zip directly; no LMDB, no full extraction.)
!python -m training.prepare_dataset prepare --zip {ZIP} --out data/seg --n-overlays 12 2>&1 | tee /content/stageA/prepare_report.txt

In [ ]:
# A5b. INDEPENDENT checks on the generated masks (these do not trust the SVG mapping):
#  (1) which of the 5 classes occur, how often, and how many images have NO door / NO window pixels
#  (2) alignment test: wall pixels should sit on the darkest image pixels at zero shift. If the mask is shifted by a few
#      pixels the mean grey under it rises, so zero shift being the minimum is evidence that image and mask line up.
import json, numpy as np, cv2
from collections import defaultdict
NAMES = ["Background", "Room", "Wall", "Door", "Window"]
lines = []
def log(s=""):
    print(s); lines.append(s)

man = json.load(open("data/seg/manifest.json"))
log(f"requested {man['requested']} | accepted {man['accepted']}")
log(f"rejections {man['rejections']}")
log(f"duplicate drops {man['duplicate_drops']}")
log(f"unmapped room names {man['unmapped_room_names']}")
log(f"official list sizes {man.get('official_list_sizes')}")
log(f"alignment hint from prepare (median wall/room grey ratio, <1 means walls darker): {man['alignment_report']}")
log()
bad_values = 0
for split in ("train", "val", "test"):
    names = man["splits"][split]
    n = len(names)
    present, frac, ign = np.zeros(5, int), np.zeros(5), 0.0
    no_door, no_win, oversized = [], [], []
    for nm in names:
        m = cv2.imread(f"data/seg/masks/{split}/{nm}.png", cv2.IMREAD_UNCHANGED)
        assert m is not None, f"unreadable mask {split}/{nm}"
        valid = m != 255
        if not set(np.unique(m[valid]).tolist()) <= {0, 1, 2, 3, 4}:
            bad_values += 1
        tot = max(int(valid.sum()), 1)
        ign += 1 - valid.sum() / m.size
        cnt = np.bincount(m[valid].ravel(), minlength=6)[:5]
        present += cnt > 0
        frac += cnt / tot
        if cnt[3] == 0: no_door.append(nm)
        if cnt[4] == 0: no_win.append(nm)
        for c in (3, 4):
            if cnt[c] / tot > 0.10:
                oversized.append((nm, NAMES[c], round(float(cnt[c] / tot), 3)))
    log(f"--- {split}: {n} images")
    for c in range(5):
        log(f"  {NAMES[c]:10s} in {present[c]:4d}/{n} images ({100 * present[c] / max(n, 1):5.1f}%) | mean share of labelled pixels {100 * frac[c] / max(n, 1):6.2f}%")
    log(f"  ignored (255) pixels: {100 * ign / max(n, 1):.2f}% of image area on average")
    log(f"  images with NO door pixels: {len(no_door)} | NO window pixels: {len(no_win)}")
    if oversized:
        log(f"  SUSPICIOUS (door/window >10% of labelled pixels): {oversized[:8]}")
log(f"masks with values outside {{0..4, 255}}: {bad_values}")

log()
log("--- alignment test (train sample): grey level under the wall mask at shifts of 0 / +-6 / +-12 px")
rng = np.random.default_rng(0)
train = man["splits"]["train"]
sample = list(rng.choice(train, size=min(40, len(train)), replace=False))
shifts = [(dy, dx) for dy in (-12, -6, 0, 6, 12) for dx in (-12, -6, 0, 6, 12)]
hits, gray_by_class, tested = 0, defaultdict(list), 0
for nm in sample:
    img = cv2.imread(f"data/seg/images/train/{nm}.png", cv2.IMREAD_GRAYSCALE).astype(np.float32)
    m = cv2.imread(f"data/seg/masks/train/{nm}.png", cv2.IMREAD_UNCHANGED)
    wall = m == 2
    if wall.sum() < 50:
        continue
    means = {s: float(img[np.roll(wall, s, axis=(0, 1))].mean()) for s in shifts}
    hits += min(means, key=means.get) == (0, 0)
    tested += 1
    for c in range(5):
        sel = m == c
        if sel.any():
            gray_by_class[c].append(float(img[sel].mean()))
log(f"zero shift gives the darkest wall pixels in {hits}/{tested} sampled images ({100 * hits / max(tested, 1):.0f}%); expected: most of them")
log("mean image grey (0 black .. 255 white) under each class, averaged over those images: " +
    ", ".join(f"{NAMES[c]} {np.mean(v):.0f}" for c, v in sorted(gray_by_class.items())))
log("expected pattern: Wall clearly darker than Room and Background")
open("/content/stageA/class_coverage.txt", "w").write("\n".join(lines) + "\n")

In [ ]:
# A6. LOOK at the overlays. Left: image. Right: ground-truth mask from the mapping (blue Room, red Wall, green Door, yellow Window, hatched grey ignored).
# Walls must sit on the drawn walls, doors/windows on the openings, rooms inside the walls. Misaligned masks = do NOT train.
import glob, matplotlib.pyplot as plt, matplotlib.image as mpimg
files = sorted(glob.glob("data/seg/overlays/train_*.png"))[:6] + sorted(glob.glob("data/seg/overlays/val_*.png"))[:2] + sorted(glob.glob("data/seg/overlays/test_*.png"))[:2]
for f in files:
    plt.figure(figsize=(16, 7)); plt.imshow(mpimg.imread(f)); plt.axis("off"); plt.title(os.path.basename(f)); plt.show()

In [ ]:
# A7. Package everything needed for an independent review, and back it up to Drive.
!cp data/seg/manifest.json /content/stageA/manifest.json
!cd /content/stageA && zip -q -r stageA_for_review.zip inspect_report.txt prepare_report.txt class_coverage.txt manifest.json && cd {REPO}/data/seg && zip -q -r /content/stageA/overlays.zip overlays
!cp /content/stageA/*.zip {DRIVE}/
from google.colab import files
files.download("/content/stageA/stageA_for_review.zip")
files.download("/content/stageA/overlays.zip")

### Gate — do not continue until the overlays have been checked
Send `stageA_for_review.zip` and `overlays.zip` to Claude (attach them in the chat). Claude will look at the overlay images and the
inspect report and say whether the mapping is right. Then set the flag below. Training refuses to start without the confirmation file.

In [ ]:
# A8. Human confirmation gate. Change to True only after YOU looked at the overlays and they line up with the images.
OVERLAYS_CONFIRMED = False
from datetime import datetime, timezone
if OVERLAYS_CONFIRMED:
    open("data/seg/overlays_confirmed.txt", "w").write(f"overlays inspected and confirmed {datetime.now(timezone.utc).isoformat(timespec='seconds')}\n")
    print("confirmation file written")
else:
    print("NOT confirmed: Stage B will refuse to train. Fix the mapping or ask Claude to review.")

## Stage B — model (only after the gate)

In [ ]:
# B1. Pre-flight: the full test suite, including the torch tests (shapes, loss, optimiser loop, train->checkpoint->predict->evaluate
# round trip on tiny fake data) and the shapely tests that could not be run in the authoring sandbox. Fix failures before training.
!python -m pytest tests -q -rs

In [ ]:
# B2. Pipeline sanity: overfit 8 real training images (train and validate on the SAME 8). This only shows that the code can learn.
# It is labelled overfit_sanity and is never a result. Loss should fall clearly and mIoU on these 8 images should rise.
!python training/train_segmentation.py --overfit 8 --epochs 30 --run-name overfit8

In [ ]:
# B3. FULL fine-tuning on the prepared training split, model selection on the validation split.
# ImageNet-pretrained ResNet-18 encoder + U-Net decoder, AdamW, CE + Dice, fp16 autocast, 512x512 letterboxed, batch 4.
# Saves runs/seg/unet_resnet18/{best.pt, history.json, config.json, train_summary.json}.
!python training/train_segmentation.py --run-name unet_resnet18

In [ ]:
# B4. Training curves from the saved history (real values read from runs/seg/unet_resnet18/history.json)
import json, matplotlib.pyplot as plt
h = json.load(open("runs/seg/unet_resnet18/history.json"))
ep = [r["epoch"] for r in h]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, [r["train_loss"] for r in h], label="train"); ax[0].plot(ep, [r["val_loss"] for r in h], label="val"); ax[0].set_title("CE + Dice loss"); ax[0].legend()
ax[1].plot(ep, [r["val_mean_iou"] for r in h]); ax[1].set_title("validation mean IoU")
plt.savefig("runs/seg/unet_resnet18/training_curves.png", dpi=150, bbox_inches="tight"); plt.show()
print(json.dumps(json.load(open("runs/seg/unet_resnet18/train_summary.json")), indent=2))

In [ ]:
# B5. Evaluate on the held-out TEST split (never used for training or model selection). Aborts if any test id was seen in training.
!python evaluation/evaluate_segmentation.py --checkpoint runs/seg/unet_resnet18/best.pt --out-dir results/segmentation --n-visuals 20 2>&1 | tee results_log.txt

In [ ]:
# B6. Look at predictions: image | ground truth | prediction (first and worst test images)
import glob, matplotlib.pyplot as plt, matplotlib.image as mpimg
for f in sorted(glob.glob("results/segmentation/visuals/*.png"))[:8]:
    plt.figure(figsize=(18, 6)); plt.imshow(mpimg.imread(f)); plt.axis("off"); plt.title(os.path.basename(f)); plt.show()

In [ ]:
# B7. Package results (small files, safe for Git) and put the weights somewhere outside Git (your Drive).
import hashlib, shutil
!cp results_log.txt results/segmentation/ 2>/dev/null; cp runs/seg/unet_resnet18/training_curves.png results/segmentation/
!cd {REPO} && zip -q -r /content/segmentation_results.zip results/segmentation
shutil.copy("runs/seg/unet_resnet18/best.pt", f"{DRIVE}/unet_resnet18_best.pt")
shutil.copy("/content/segmentation_results.zip", f"{DRIVE}/segmentation_results.zip")
sha = hashlib.sha256(open("runs/seg/unet_resnet18/best.pt", "rb").read()).hexdigest()
print("weights saved to", f"{DRIVE}/unet_resnet18_best.pt", "| bytes:", os.path.getsize("runs/seg/unet_resnet18/best.pt"), "\nsha256:", sha)
from google.colab import files
files.download("/content/segmentation_results.zip")

### Send back
Attach `segmentation_results.zip` (and, if you want the model in the repo's release, tell Claude where the weights live).
Claude will re-check the numbers against the raw `metrics.json`, update the README and slides from those files only, and push.